# PartB_1

This notebook implements transfer learning by using a pretrained CNN as a feature extractor and pairing its output with standard machine learning classifiers.

Structure:
1. Utilities and data loading
2. Pretrained feature extractor
3. Secondary machine learning variants and benchmark

## Colab Setup

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")
#!unzip "/content/gdrive/MyDrive/Deep_learning_2/flower.h5.zip" -d "/content/gdrive/MyDrive/Deep_learning_2/"
!ls /content/gdrive/MyDrive/Deep_learning_2


## Utilities

This section contains the imports, the HDF5 loader, the visual evaluation utilities, and the comparison plotting functions.

In [ ]:
import csv
import time
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# source: https://www.tensorflow.org/tutorials/images/transfer_learning
# source: https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2
# source: https://www.tensorflow.org/api_docs/python/tf/keras/applications/mobilenet_v2/preprocess_input
# source: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
# source: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# source: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html


In [ ]:
def loadDataH5():
    candidate_paths = [
        "data1.h5",
        "/content/data1.h5",
        "/content/gdrive/MyDrive/Deep_learning_2/data1.h5",
        "/content/drive/MyDrive/Deep_learning_2/data1.h5",
    ]

    data_path = None
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            data_path = candidate
            break

    if data_path is None:
        raise FileNotFoundError(
            "Could not find data1.h5. Expected it in the current directory, "
            "/content, or /content/gdrive/MyDrive/Deep_learning_2."
        )

    print("Using data file:", data_path)

    with h5py.File(data_path, "r") as hf:
        trainX = np.array(hf.get("trainX"))
        trainY = np.array(hf.get("trainY"))
        valX = np.array(hf.get("valX"))
        valY = np.array(hf.get("valY"))

    print("trainX shape:", trainX.shape, "trainY shape:", trainY.shape)
    print("valX shape:", valX.shape, "valY shape:", valY.shape)
    return trainX, trainY, valX, valY


def plot_confusion_matrix(y_true, y_pred, model_name, class_names, output_dir="plots_partB1"):
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(
        cm.astype("float"),
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
    ax.set_title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_normalized,
        display_labels=class_names,
    )
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True, values_format=".2f")
    ax.set_title(f"{model_name} Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.csv"),
        cm,
        delimiter=",",
        fmt="%d",
    )
    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.csv"),
        cm_normalized,
        delimiter=",",
        fmt="%.6f",
    )

    return cm, cm_normalized


def save_classification_report(y_true, y_pred, class_names, model_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    txt_path = os.path.join(output_dir, f"{model_name}_classification_report.txt")
    with open(txt_path, "w") as report_file:
        report_file.write(report_text)

    csv_path = os.path.join(output_dir, f"{model_name}_classification_report.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["label", "precision", "recall", "f1-score", "support"])
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                writer.writerow([
                    label,
                    metrics.get("precision"),
                    metrics.get("recall"),
                    metrics.get("f1-score"),
                    metrics.get("support"),
                ])


def plot_prediction_examples(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class=None,
    max_images=6,
    model_name="model",
    output_dir="plots_partB1",
):
    os.makedirs(output_dir, exist_ok=True)

    if pred_class is None:
        indices = np.where((y_true == true_class) & (y_pred == true_class))[0]
        title = f"{model_name}: correctly classified class {true_class}"
        file_name = f"{model_name}_class_{true_class}_correct_examples.png"
    else:
        indices = np.where((y_true == true_class) & (y_pred == pred_class))[0]
        title = f"{model_name}: class {true_class} misclassified as class {pred_class}"
        file_name = f"{model_name}_class_{true_class}_pred_{pred_class}_examples.png"

    if len(indices) == 0:
        print(f"No matching examples found for {title}.")
        return

    indices = indices[:max_images]
    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx])
        ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, file_name), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def get_top_confusion_pair(cm):
    confusion_only = cm.copy()
    np.fill_diagonal(confusion_only, 0)
    max_index = np.argmax(confusion_only)
    true_class, pred_class = np.unravel_index(max_index, confusion_only.shape)
    if confusion_only[true_class, pred_class] == 0:
        return None
    return true_class, pred_class


def plot_class_confusion_comparison(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class,
    max_images=4,
    model_name="model",
    output_dir="plots_partB1",
):
    os.makedirs(output_dir, exist_ok=True)

    misclassified_idx = np.where((y_true == true_class) & (y_pred == pred_class))[0]
    true_correct_idx = np.where((y_true == true_class) & (y_pred == true_class))[0]
    pred_correct_idx = np.where((y_true == pred_class) & (y_pred == pred_class))[0]

    if len(misclassified_idx) == 0:
        print(
            f"No misclassified examples found for class {true_class} predicted as class {pred_class}."
        )
        return

    misclassified_idx = misclassified_idx[:max_images]
    true_correct_idx = true_correct_idx[:max_images]
    pred_correct_idx = pred_correct_idx[:max_images]

    columns = max(len(misclassified_idx), len(true_correct_idx), len(pred_correct_idx), 1)
    fig, axes = plt.subplots(3, columns, figsize=(3 * columns, 9))

    if columns == 1:
        axes = np.array(axes).reshape(3, 1)

    row_titles = [
        f"Misclassified: true={true_class}, pred={pred_class}",
        f"Correct examples of true class {true_class}",
        f"Correct examples of predicted class {pred_class}",
    ]
    row_indices = [misclassified_idx, true_correct_idx, pred_correct_idx]

    for row, (title, indices) in enumerate(zip(row_titles, row_indices)):
        for col in range(columns):
            ax = axes[row, col]
            if col < len(indices):
                idx = indices[col]
                ax.imshow(images[idx])
                ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
                ax.axis("off")
            else:
                ax.axis("off")
        axes[row, 0].set_ylabel(title, rotation=90, fontsize=11, labelpad=20)

    plt.suptitle(
        f"{model_name}: visual comparison for confusion {true_class} -> {pred_class}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            output_dir,
            f"{model_name}_class_{true_class}_vs_class_{pred_class}_comparison.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


def plot_variant_comparison(results, output_dir="plots_partB1"):
    os.makedirs(output_dir, exist_ok=True)
    model_names = list(results.keys())
    accuracies = [results[name]["val_accuracy"] for name in model_names]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(model_names, accuracies, color=["#4C72B0", "#55A868", "#C44E52"])
    ax.set_title("PartB_1 Validation Accuracy Comparison")
    ax.set_xlabel("Secondary Classifier")
    ax.set_ylabel("Validation Accuracy")
    ax.set_ylim(0, 1)

    for index, accuracy in enumerate(accuracies):
        ax.text(index, accuracy + 0.01, f"{accuracy:.3f}", ha="center")

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, "partB1_variant_accuracy_comparison.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


## Feature Extractor

This section uses a pretrained MobileNetV2 network as a fixed feature extractor, following the TensorFlow transfer learning tutorial. The CNN is not used as the final classifier. Instead, its output feature vectors are used as input to standard machine learning algorithms.


In [ ]:
def build_feature_extractor(input_shape=(128, 128, 3)):
    return tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )


def extract_features(feature_extractor, images, batch_size=32):
    # The assignment data is normalized to [0, 1], while MobileNetV2 preprocessing
    # expects image-like input. We scale back to [0, 255] before preprocessing.
    images_for_model = tf.keras.applications.mobilenet_v2.preprocess_input(images * 255.0)
    features = feature_extractor.predict(images_for_model, batch_size=batch_size, verbose=1)
    print("Extracted feature shape:", features.shape)
    return features


## Secondary Classifiers and Benchmark

This section compares three standard machine learning algorithms on the extracted deep features:
- Logistic Regression
- Random Forest
- K-Nearest Neighbors

In [ ]:
def build_secondary_models():
    return {
        "logisticRegression": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(max_iter=3000, random_state=42)),
            ]
        ),
        "randomForest": RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        ),
        "knn": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("classifier", KNeighborsClassifier(n_neighbors=5)),
            ]
        ),
    }


def evaluate_model(images, y_true, y_pred, model_name, output_dir="plots_partB1"):
    class_names = [f"Class {index}" for index in range(17)]
    cm, cm_normalized = plot_confusion_matrix(
        y_true, y_pred, model_name, class_names, output_dir=output_dir
    )

    print(f"\nClassification report for {model_name}:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    save_classification_report(y_true, y_pred, class_names, model_name, output_dir)

    best_class = int(np.argmax(np.diag(cm_normalized)))
    best_class_score = np.diag(cm_normalized)[best_class]
    print(
        f"Best classified class for {model_name}: "
        f"class {best_class} with normalized recall {best_class_score:.2f}"
    )

    plot_prediction_examples(
        images,
        y_true,
        y_pred,
        true_class=best_class,
        pred_class=None,
        max_images=5,
        model_name=model_name,
        output_dir=output_dir,
    )

    top_confusion = get_top_confusion_pair(cm)
    if top_confusion is not None:
        true_class, pred_class = top_confusion
        plot_prediction_examples(
            images,
            y_true,
            y_pred,
            true_class=true_class,
            pred_class=pred_class,
            max_images=5,
            model_name=model_name,
            output_dir=output_dir,
        )
        plot_class_confusion_comparison(
            images,
            y_true,
            y_pred,
            true_class=true_class,
            pred_class=pred_class,
            max_images=4,
            model_name=model_name,
            output_dir=output_dir,
        )


def run_variants(train_features, trainY, val_features, valY, val_images):
    output_dir = "plots_partB1"
    os.makedirs(output_dir, exist_ok=True)

    models = build_secondary_models()
    results = {}

    for model_name, model in models.items():
        print(f"\n{'=' * 60}")
        print(f"Training {model_name}")
        print(f"{'=' * 60}")

        fit_start = time.perf_counter()
        model.fit(train_features, trainY)
        fit_seconds = time.perf_counter() - fit_start
        predict_start = time.perf_counter()
        y_pred = model.predict(val_features)
        predict_seconds = time.perf_counter() - predict_start
        val_accuracy = accuracy_score(valY, y_pred)

        results[model_name] = {
            "val_accuracy": val_accuracy,
            "y_pred": y_pred,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "predict_ms_per_image": (predict_seconds / len(valY)) * 1000,
        }

        print(f"{model_name} validation accuracy: {val_accuracy:.4f}")
        print(f"{model_name} training time: {fit_seconds:.4f}s")
        print(f"{model_name} prediction time: {predict_seconds:.4f}s ({(predict_seconds / len(valY)) * 1000:.3f} ms/image)")
        evaluate_model(val_images, valY, y_pred, model_name, output_dir=output_dir)

    plot_variant_comparison(results, output_dir=output_dir)

    print("\nFinal results summary:")
    for model_name, result in results.items():
        print(
            f"{model_name}: val_accuracy={result['val_accuracy']:.4f}, "
            f"train_time={result['fit_seconds']:.4f}s, "
            f"predict_time={result['predict_seconds']:.4f}s"
        )

    best_model_name = max(results, key=lambda name: results[name]["val_accuracy"])
    print(
        f"\nBest variant: {best_model_name} "
        f"with validation accuracy {results[best_model_name]['val_accuracy']:.4f}"
    )

    return results


## Run the Benchmark

Execute the following cell to extract features with MobileNetV2 and compare the three secondary classifiers.


In [ ]:
trainX, trainY, valX, valY = loadDataH5()
feature_extractor = build_feature_extractor(input_shape=(128, 128, 3))

train_features = extract_features(feature_extractor, trainX)
val_features = extract_features(feature_extractor, valX)

results = run_variants(train_features, trainY, val_features, valY, valX)

## Export Results to CSV

Export the main metrics to CSV after running the benchmark.

In [ ]:
import csv
import os
import shutil
import time


def export_partb1_results(results, output_dir="exports_partB1", copy_to_drive=True):
    os.makedirs(output_dir, exist_ok=True)

    summary_path = os.path.join(output_dir, "partB1_summary.csv")
    with open(summary_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["model_name", "val_accuracy", "fit_seconds", "predict_seconds", "predict_ms_per_image"])
        for model_name, metrics in results.items():
            writer.writerow([
                model_name,
                metrics.get("val_accuracy"),
                metrics.get("fit_seconds"),
                metrics.get("predict_seconds"),
                metrics.get("predict_ms_per_image"),
            ])

    master_path = os.path.join(output_dir, "partB1_master_summary.csv")
    with open(master_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["question", "model_name", "val_loss", "val_accuracy", "training_seconds", "predict_seconds", "predict_ms_per_image", "short_comment"])
        for model_name, metrics in results.items():
            writer.writerow(["PartB_1", model_name, "N/A", metrics.get("val_accuracy"), metrics.get("fit_seconds"), metrics.get("predict_seconds"), metrics.get("predict_ms_per_image"), ""])

    print(f"Saved CSV file to {output_dir}")
    print(summary_path)

    if copy_to_drive:
        drive_targets = [
            "/content/gdrive/MyDrive/Deep_learning_2",
            "/content/drive/MyDrive/Deep_learning_2",
        ]
        copied = False
        for drive_dir in drive_targets:
            if os.path.isdir(drive_dir):
                drive_export_dir = os.path.join(drive_dir, output_dir)
                os.makedirs(drive_export_dir, exist_ok=True)
                shutil.copy2(summary_path, os.path.join(drive_export_dir, os.path.basename(summary_path)))
                print(f"Copied CSV file to {drive_export_dir}")
                plot_source_dir = "plots_partB1"
                if os.path.isdir(plot_source_dir):
                    drive_plot_dir = os.path.join(drive_dir, plot_source_dir)
                    os.makedirs(drive_plot_dir, exist_ok=True)
                    for file_name in os.listdir(plot_source_dir):
                        source_file = os.path.join(plot_source_dir, file_name)
                        if os.path.isfile(source_file):
                            shutil.copy2(source_file, os.path.join(drive_plot_dir, file_name))
                    print(f"Copied plot files to {drive_plot_dir}")
                copied = True
                break
        if not copied:
            print("Google Drive export folder not found. CSV file was saved locally only.")


if "results" in globals():
    export_partb1_results(results)
else:
    print("Run the benchmark cell first, then rerun this export cell.")
